In [ ]:
%sql
/* ===================================================================================
   DIAGNOSTIC: Find Missing Master AUs in CC Mapping
   
   PURPOSE: Identifies the difference between the 61 target AUs in the Master List 
   and the 59 AUs found in the Cost Center Mapping Bootstrap.
=================================================================================== */

WITH Master_AUs AS (
    -- The 61 target AUs
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(`US OR Canada`)) = 'CANADA'
),

CC_Mapped_AUs AS (
    -- The AUs present in the Cost Center mapping
    SELECT DISTINCT TRIM(CAST(AU_ID AS STRING)) AS Mapped_AU_ID
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL
)

-- Find the ones in the Master List that do NOT exist in the CC Mapping
SELECT 
    m.BusinessID AS Missing_AU_ID,
    m.AU_Name AS Missing_AU_Name,
    'Missing from CC Mapping' AS Reason
FROM Master_AUs m
LEFT JOIN CC_Mapped_AUs c
    ON m.BusinessID = c.Mapped_AU_ID
WHERE c.Mapped_AU_ID IS NULL
ORDER BY m.BusinessID;